# Chapter 7A — Self-Attention: Worked Example

> A step-by-step trace of scaled dot-product attention using a concrete example.
> Every shape, every number, every operation shown explicitly.

**Setup:**
```
string  : "First principles"
B       : 4 batches
T       : 4 tokens per batch
d_model : 5  (embedding size per token)
d_k     : 8  (query / key / value projection size)
```

**Batches:**
```
Batch 0 : ['F', 'i', 'r', 's']
Batch 1 : ['t', ' ', 'p', 'r']
Batch 2 : ['i', 'n', 'c', 'i']
Batch 3 : ['p', 'l', 'e', 's']
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

# ── parameters ────────────────────────────────────────────────────────────────
B       = 4   # number of batches
T       = 4   # tokens per batch (context window)
d_model = 5   # embedding dimension per token
d_k     = 8   # query / key / value dimension

# ── the string and its batches ────────────────────────────────────────────────
string  = "First principles"
batches = [string[i*T:(i+1)*T] for i in range(B)]

# ── vocabulary built from the string ─────────────────────────────────────────
chars      = sorted(set(string))
vocab_size = len(chars)
stoi       = {c: i for i, c in enumerate(chars)}

print(f'String     : {repr(string)}')
print(f'B={B}, T={T}, d_model={d_model}, d_k={d_k}')
print(f'Vocab size : {vocab_size} unique characters: {chars}')
print()
for i, b in enumerate(batches):
    print(f'  Batch {i}: {list(b)}')

## Step 1 — Token Embeddings

Each character maps to a row in a learned embedding table (`nn.Embedding`).
Repeated characters — `'i'` appears three times in "First principles" — share the same embedding row
and therefore receive identical vectors before projection.

```
X shape: (B=4, T=4, d_model=5)
```

In [ ]:
# ── token embeddings via nn.Embedding ─────────────────────────────────────────
emb_table = nn.Embedding(vocab_size, d_model)

# Integer token IDs for each batch
batch_ids = torch.tensor([[stoi[c] for c in b] for b in batches])  # (B, T)
X         = emb_table(batch_ids)                                     # (B, T, d_model)

print(f'batch_ids shape: {list(batch_ids.shape)}  (integer token IDs)')
print(f'X shape        : {list(X.shape)}  = (B, T, d_model)')
print()
print('Batch 0 embeddings (one row = one token, 5 numbers each):')
for t, ch in enumerate(batches[0]):
    vals = '  '.join(f'{v:+.3f}' for v in X[0, t].detach())
    print(f'  pos {t} ({ch!r}): [{vals}]')
print()
# Verify repeated characters share the same embedding
# 'i' appears at batch0/pos1, batch2/pos0, batch2/pos3
i_emb_b0 = X[0, 1].detach()   # 'i' in batch 0
i_emb_b2 = X[2, 0].detach()   # 'i' in batch 2
print(f'"i" embedding (batch 0, pos 1) == (batch 2, pos 0): {torch.allclose(i_emb_b0, i_emb_b2)}')
print('Same character → same embedding row (before projection).')

## Step 2 — Weight Matrices W_Q, W_K, W_V

Three learned weight matrices project each token embedding from `d_model=5` → `d_k=8`.

```
W_Q : (d_model=5, d_k=8)   "what am I looking for?"
W_K : (d_model=5, d_k=8)   "what do I contain?"
W_V : (d_model=5, d_k=8)   "what do I contribute?"
```

**Key facts:**
- All three have the same shape
- All three trained simultaneously in one training loop
- Shape has NOTHING to do with vocab size — only d_model and d_k
- Same W applied to all B batches
- In this example d_k > d_model; in larger models d_k is typically a fraction of d_model

In [ ]:
# ── weight matrices ───────────────────────────────────────────────────────────
W_Q = torch.randn(d_model, d_k)   # (5, 8)
W_K = torch.randn(d_model, d_k)   # (5, 8)
W_V = torch.randn(d_model, d_k)   # (5, 8)

print(f'W_Q shape: {list(W_Q.shape)}  = (d_model, d_k)  — {W_Q.numel()} parameters')
print(f'W_K shape: {list(W_K.shape)}  = (d_model, d_k)  — {W_K.numel()} parameters')
print(f'W_V shape: {list(W_V.shape)}  = (d_model, d_k)  — {W_V.numel()} parameters')
print(f'Total Q/K/V parameters: {W_Q.numel() + W_K.numel() + W_V.numel()}')
print()
print('These are shared across all 4 batches.')
print('Batch dimension = parallelism, not separate weights.')

## Step 3 — Compute Q, K, V

Matrix multiply X by each weight matrix:

```
Q = X @ W_Q : (4, 4, 5) @ (5, 8) → (4, 4, 8)  = (B, T, d_k)
K = X @ W_K : (4, 4, 5) @ (5, 8) → (4, 4, 8)
V = X @ W_V : (4, 4, 5) @ (5, 8) → (4, 4, 8)
```

PyTorch applies W to every (T, d_model) slice automatically — one W_Q for all 4 batches.

In [ ]:
# ── projections ───────────────────────────────────────────────────────────────
X_d = X.detach()   # detach from embedding graph for manual ops
Q = X_d @ W_Q      # (4, 4, 8)
K = X_d @ W_K      # (4, 4, 8)
V = X_d @ W_V      # (4, 4, 8)

print(f'Q shape: {list(Q.shape)}  = (B, T, d_k)')
print(f'K shape: {list(K.shape)}')
print(f'V shape: {list(V.shape)}')
print()
print('Batch 0, token F (pos 0) query vector:')
print(f'  q_F = [{"  ".join(f"{v:+.3f}" for v in Q[0,0])}]')
print(f'  shape: {list(Q[0,0].shape)}  (8 numbers — the query space)')

## Step 4 — Score Matrix: Q @ K.T

For each batch, compute how much every query matches every key:

```
scores[b, t, s] = q_t · k_s
                = "how relevant is position s to position t?"

Q   : (4, 4, 8)
K.T : (4, 8, 4)   ← transpose last two dims
scores = Q @ K.T → (4, 4, 4) = (B, T, T)
```

**Total dot products: B × T × T = 4 × 4 × 4 = 64** — all in one matrix operation.

**This is why attention is O(T²)** — the score matrix grows as T×T.

Note: the dot product measures compatibility through both vector direction and magnitude.
It is not normalised cosine similarity.

In [ ]:
# ── raw scores ────────────────────────────────────────────────────────────────
scores_raw = Q @ K.transpose(-2, -1)   # (4, 4, 4)

print(f'scores shape: {list(scores_raw.shape)}  = (B, T, T)')
print()
print('Batch 0 score matrix (rows=queries, cols=keys):')
header = '       ' + ''.join(f'  {ch}   ' for ch in batches[0])
print(header)
for t, ch in enumerate(batches[0]):
    row = '  '.join(f'{scores_raw[0,t,s].item():+.3f}' for s in range(T))
    print(f'  {ch!r} (q{t}): [{row}]')
print()
print(f'scores_raw[0, 0, 1] = {scores_raw[0,0,1].item():+.4f}')
print(f'  = q_F · k_i  = how relevant is "i" to "F"')

## Step 5 — Scale by √d_k

Divide every score by `√d_k`.

**Why?**
```
q · k = Σ q_i * k_i   (sum of d_k terms)

If components are zero-mean, unit-variance, independent:
  Var(q · k) = d_k   →   Std = √d_k

Large scores → highly saturated softmax
            → very small gradients for low-weight positions
            → those positions stop learning

Dividing by √d_k normalises std back to ~1.
```

This is an initialisation-motivated approximation (Vaswani et al. 2017, Section 3.2.1),
not a guarantee about trained distributions.

The demonstration below uses variance-controlled random vectors at a larger d_k
so the effect is clearly visible.

In [ ]:
# ── scale by sqrt(d_k) ────────────────────────────────────────────────────────
scale  = math.sqrt(d_k)
scores = scores_raw / scale

print(f'√d_k = √{d_k} = {scale:.4f}')
print(f'Before scaling — std of scores (batch 0): {scores_raw[0].std().item():.4f}')
print(f'After  scaling — std of scores (batch 0): {scores[0].std().item():.4f}')
print()

# ── controlled demonstration: larger d_k shows the effect clearly ─────────────
# Use variance-controlled init (std=1/√d_k) and multiple seeds to average out noise.
torch.manual_seed(0)
T_demo, d_k_demo = 8, 64
entropy = lambda w: (-(w.clamp(min=1e-9).log() * w).sum(dim=-1)).mean().item()

max_raw_list, max_scaled_list = [], []
ent_raw_list, ent_scaled_list = [], []

for _ in range(10):   # average over 10 random seeds
    q = torch.randn(T_demo, d_k_demo)
    k = torch.randn(T_demo, d_k_demo)
    s_raw    = q @ k.T
    s_scaled = s_raw / math.sqrt(d_k_demo)
    w_raw    = F.softmax(s_raw,    dim=-1)
    w_scaled = F.softmax(s_scaled, dim=-1)
    max_raw_list.append(w_raw.max(dim=-1).values.mean().item())
    max_scaled_list.append(w_scaled.max(dim=-1).values.mean().item())
    ent_raw_list.append(entropy(w_raw))
    ent_scaled_list.append(entropy(w_scaled))

import statistics
print(f'Scaling demonstration (T={T_demo}, d_k={d_k_demo}, avg over 10 seeds):')
print(f'  Without scaling — mean max weight: {statistics.mean(max_raw_list):.4f}   mean entropy: {statistics.mean(ent_raw_list):.4f}')
print(f'  With scaling    — mean max weight: {statistics.mean(max_scaled_list):.4f}   mean entropy: {statistics.mean(ent_scaled_list):.4f}')
print()
print('Higher entropy = attention distributed across more positions.')
print('Without scaling: one position dominates, gradient signal concentrates there.')
print('With scaling:    attention spreads, all positions receive gradient signal.')

## Step 6 — Causal Mask

Language modelling is causal: position t can only attend to positions 0..t.
Future positions are masked to `-inf` before softmax → `exp(-inf) = 0` exactly.

Causal masking introduces an ordering constraint, but it does not supply full
positional information. Chapter 7B adds learned positional embeddings for that.

```
mask (T=4):
        s=0  s=1  s=2  s=3
t=0  [  1    0    0    0  ]   'F' sees only itself
t=1  [  1    1    0    0  ]   'i' sees F, i
t=2  [  1    1    1    0  ]   'r' sees F, i, r
t=3  [  1    1    1    1  ]   's' sees all
```

In [ ]:
# ── causal mask ───────────────────────────────────────────────────────────────
mask           = torch.tril(torch.ones(T, T, dtype=torch.bool))   # (4, 4)
scores_masked  = scores.masked_fill(~mask, float('-inf'))          # (4, 4, 4)

print('Causal mask:')
for t in range(T):
    row = '  '.join('1' if mask[t,s].item() else '0' for s in range(T))
    print(f'  pos {t} ({batches[0][t]!r}): [{row}]')
print()
print('Batch 0 scores after masking:')
for t in range(T):
    row = '  '.join(
        f'{scores_masked[0,t,s].item():+.3f}' if mask[t,s] else '  -inf'
        for s in range(T)
    )
    print(f'  pos {t} ({batches[0][t]!r}): [{row}]')

## Step 7 — Softmax → Attention Weights

Apply softmax row-wise. Each row sums to 1.0.
`exp(-inf) = 0` — masked positions receive exactly zero weight.

In [ ]:
# ── softmax ───────────────────────────────────────────────────────────────────
weights = F.softmax(scores_masked, dim=-1)   # (4, 4, 4)

print(f'weights shape: {list(weights.shape)}  = (B, T, T)')
print()
print('Batch 0 attention weights (each row sums to 1.0):')
header = '       ' + ''.join(f'  {ch}    ' for ch in batches[0])
print(header)
for t in range(T):
    row   = '  '.join(f'{weights[0,t,s].item():.4f}' for s in range(T))
    total = weights[0,t].sum().item()
    print(f'  pos {t} ({batches[0][t]!r}): [{row}]  sum={total:.4f}')
print()
print('Upper triangle = 0.0000 exactly (causal mask working).')

## Step 8 — Output: weights @ V

Weighted sum of value vectors:

```
output[t] = Σ_s  weights[t, s] × V[s]

weights : (B=4, T=4, T=4)
V       : (B=4, T=4, d_k=8)
output  : (B=4, T=4, d_k=8)
```

Each position's output is a content-dependent blend of all value vectors.
Compare to Ch 6: each position's output was a fixed transformation of its own embedding only.

In [ ]:
# ── weighted value aggregation ────────────────────────────────────────────────
output = weights @ V   # (4, 4, 8)

print(f'output shape: {list(output.shape)}  = (B, T, d_k)')
print()
print('Batch 0 output vectors (one row = one position, 8 numbers):')
for t in range(T):
    ch   = batches[0][t]
    vals = '  '.join(f'{v:+.3f}' for v in output[0, t])
    print(f'  pos {t} ({ch!r}): [{vals}]')
print()
print('Position 3 ("s") — actual blend of value vectors (from this notebook run):')
for s in range(T):
    w = weights[0, 3, s].item()
    print(f'  {w:.4f} × V[{s}] ({batches[0][s]!r})')
print(f'  → [{"  ".join(f"{v:+.3f}" for v in output[0,3])}]')

## Summary — Full Pipeline

```
batch_ids (B=4, T=4)       — integer token IDs
        │
   nn.Embedding             — lookup table (vocab_size, d_model)
        │
X (B=4, T=4, d_model=5)    — token embeddings
        │
   ┌────┴─────┐
  W_Q        W_K        W_V       (each 5×8)
   │          │          │
   Q          K          V        (each B,T,d_k = 4,4,8)
   │          │
   └──Q@K.T──┘
        │
    / √d_k                        (4,4,4) score matrix
        │
   causal mask                    upper triangle → -inf
        │
     softmax                      rows sum to 1
        │
   weights @ V                    (4,4,8) context-aware output
```

**Total score computations:** B × T × T = 4 × 16 = 64 dot products  
**This is O(T²)** — the fundamental cost of attention.

In [ ]:
# ── full pipeline in one block ────────────────────────────────────────────────
def scaled_dot_product_attention(X, W_Q, W_K, W_V):
    B, T, _ = X.shape
    Q       = X @ W_Q
    K       = X @ W_K
    V       = X @ W_V
    scores  = Q @ K.transpose(-2, -1) / math.sqrt(W_Q.shape[-1])
    mask    = torch.tril(torch.ones(T, T, dtype=torch.bool))
    scores  = scores.masked_fill(~mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

out, w = scaled_dot_product_attention(X_d, W_Q, W_K, W_V)

print(f'Input  shape: {list(X_d.shape)}')
print(f'Output shape: {list(out.shape)}')
print(f'Weight shape: {list(w.shape)}')
print()
match = torch.allclose(out, output)
print(f'Matches step-by-step calculation: {match}')